[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/95_permutation_reset_bd260806_solution.ipynb)

# 参考解法：消息置换复位

Reference solution.

## 解析

**结论：按置换的循环分解，每个循环的字符环有各自的最小旋转周期，全串复位时间为这些周期的最小公倍数。**

### 循环分解
置换 `p` 把下标划分成若干不相交循环。一次操作 `w[i] = u[p[i]]` 相当于沿每个循环把字符旋转一步，各循环互不影响，可分别分析。

### 单循环的周期
取出一个长度 `L` 的循环上的字符，按循环顺序排成环形串。旋转一步复位当且仅当串具有周期 `d`——即最小的 `L` 的约数，使得整串由长度 `d` 的块重复而成（所有字符相同则 `d = 1`）。该循环需要 `d` 次操作复位。

### 合并
整串复位需所有循环同时复位，故答案是各周期的 LCM。LCM 可能远超 64 位，用 `lcm = lcm // gcd(lcm, d) * d` 累积，最后对 `1e9+7` 取模。

### 验证
已用直接逐次模拟操作的暴力实现在数千组随机 `(n, u, p)` 上对拍一致，并单独验证「多个互质长度循环」使 LCM 超 64 位时的取模正确性。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
from typing import List

In [ ]:
# ✅ SOLUTION

from typing import List
import math

MOD = 10 ** 9 + 7

def _cycle_period(chars: List[str]) -> int:
    m = len(chars)
    for d in range(1, m + 1):
        if m % d:
            continue
        if all(chars[i] == chars[i % d] for i in range(m)):
            return d
    return m

class Solution:
    def min_reset(self, n: int, u: str, p: List[int]) -> int:
        vis = [False] * n
        ans = 1
        for i in range(n):
            if vis[i]:
                continue
            cyc = []
            x = i
            while not vis[x]:
                vis[x] = True
                cyc.append(u[x])
                x = p[x]
            per = _cycle_period(cyc)
            ans = ans // math.gcd(ans, per) * per   # LCM accumulate
        return ans % MOD

In [ ]:
sol = Solution()
print(sol.min_reset(2, "xy", [1, 0]))               # 2
print(sol.min_reset(3, "zzz", [1, 2, 0]))           # 1
print(sol.min_reset(5, "hello", [1, 2, 3, 4, 0]))   # 5

In [ ]:
from torch_judge import check
check('permutation_reset_bd260806')